# Desafio 1

In [1]:
import boto3
import os
from datetime import datetime
from botocore.exceptions import ClientError

# Configuración - EDITA ESTAS
LOCAL_DIR = '/ruta/a/tu/carpeta/local'  # Carpeta local a respaldar
BUCKET_NAME = 'bucket-lfrf'             # Bucket ajustado
REGION = 'us-east-1'                    # Región AWS (ajusta si necesitas)
S3_PREFIX = 'backup/'                   # Prefijo en S3

def create_bucket_if_not_exists(s3_client, bucket_name, region):
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f"Bucket '{bucket_name}' ya existe.")
    except ClientError:
        print(f"Creando bucket '{bucket_name}' en {region}...")
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region if region != 'us-east-1' else None}
        )
        print(f"Bucket '{bucket_name}' creado exitosamente.")

def backup_to_s3():
    s3_client = boto3.client('s3', region_name=REGION)
    
    # Crea bucket si no existe
    create_bucket_if_not_exists(s3_client, BUCKET_NAME, REGION)
    
    hoy = datetime.now().strftime('%Y-%m-%d')
    fecha_prefix = f"{S3_PREFIX}{hoy}/"
    
    print(f"Iniciando backup de {LOCAL_DIR} a s3://{BUCKET_NAME}/{fecha_prefix}")
    
    archivos_subidos = 0
    for root, dirs, files in os.walk(LOCAL_DIR):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, LOCAL_DIR).replace(os.sep, '/')
            s3_key = f"{fecha_prefix}{relative_path}"
            
            try:
                s3_client.upload_file(local_path, BUCKET_NAME, s3_key)
                print(f"Subido: {local_path} -> {s3_key}")
                archivos_subidos += 1
            except Exception as e:
                print(f"Error subiendo {local_path}: {e}")
    
    print(f"Backup completado. {archivos_subidos} archivos subidos.")

if __name__ == "__main__":
    backup_to_s3()


Bucket 'bucket-lfrf' ya existe.
Iniciando backup de /ruta/a/tu/carpeta/local a s3://bucket-lfrf/backup/2026-03-18/
Backup completado. 0 archivos subidos.
